In [6]:
import torch
import torch.nn as nn
from torchvision import models

model=models.mobilenet_v2(weights='DEFAULT')
for param in model.parameters():
  param.requires_grad=False

num_ftrs=model.classifier[1].in_features

model.classifier=nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(num_ftrs, 128),
    nn.ReLU(),
    nn.Linear(128, 2) # 2 classes: Mask, No Mask
)

model.load_state_dict(torch.load('mask_detector_model.pth', map_location=torch.device('cpu')))
model.eval()

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [7]:
%pip install torch torchvision torchaudio

Note: you may need to restart the kernel to use updated packages.


In [8]:
from torchvision import transforms
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [9]:
import cv2
import torch
from PIL import Image

cap = cv2.VideoCapture(0) #Used to open camera
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml') #Loading haarcascade model

model.eval()
while True:
    ret, frame = cap.read() #frame-->each every frame captured by webcam
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) #gray scaling because haar cascade model works best there
    faces = face_cascade.detectMultiScale(gray, 1.1, 4) #detecting face. faces=list of tuples [(x,y,w,h)]

    for (x, y, w, h) in faces:
        # 1. Crop
        face_roi = frame[y:y+h, x:x+w]
        face_rgb = cv2.cvtColor(face_roi, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(face_rgb)

        # 2. Preprocess
        input_tensor = preprocess(pil_img).unsqueeze(0)

        # 3. Predict
        with torch.no_grad():
            output = model(input_tensor)
            _, pred = torch.max(output, 1)

        # 4. Annotate
        label = "Mask" if pred.item() == 0 else "No Mask"
        color = (0, 255, 0) if label == "Mask" else (0, 0, 255)
        
        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.putText(frame, label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    # 5. Display
    cv2.imshow('LIVE Mask Detection (Press q to quit)', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()